In [1]:
from src.models.cvmodels import FeatureBasedEstimator
from src.models.feature_extractors import RapidTextExtractor, OrbFeatureExtractor, SIFTFeatureExtractor
from src.models.deepmodels import SiameseDino
from transformers import AutoImageProcessor, AutoModel
from hydra import initialize, compose
import torch

with initialize(config_path="../config", version_base=None):
    cfg = compose(config_name="base_config")

checkpoint = torch.load("../model_checkpoints/sandy-fire-218.pth")

backbone_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(backbone_name)
backbone = AutoModel.from_pretrained(backbone_name, dtype=torch.float32)

model1 = SiameseDino(backbone, processor, cfg)
model1.load_state_dict(checkpoint)

rapidocr = RapidTextExtractor()
orb = OrbFeatureExtractor(1000)
sift = SIFTFeatureExtractor()
feature_extractors = [orb]
weights = [1]
model2 = FeatureBasedEstimator(feature_extractors, weights)
feature_extractors = [orb, rapidocr]
weights = [0.5, 0.5]
model3 = FeatureBasedEstimator(feature_extractors, weights)
feature_extractors = [sift]
model4 = FeatureBasedEstimator(feature_extractors, [1])

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from src.utils import ModelReport
from src.eval import Recall

metric = Recall()
modelreport = ModelReport({ "sift": model4})#"siamesedino": model1, "orb": model2, "0.5_orb+0.5_rapidocr": model3, "sift": model4})
df = modelreport.generate_report("../data/original_data_color_invariant", "random", metric)

Found 88 total classes.
Processing Train classes...
Processing Validation classes...
Processing Test classes...

--- Data Split Summary ---
Training Loader:      16 samples from 1 classes (Augmented)
Gallery Loader:      265 samples from 88 classes (Original)
Val Query Loader:     87 samples from 87 classes (Original)
Test Query Loader:     0 samples from 0 classes (Original)
Loading dataloaders...
🔨 Preparing gallery (computing embeddings)...
✅ Gallery prepared: 264 images
Elapsed time for evaluate = 3.18s
Elapsed time for inference = 0.19s
Elapsed time for find_nearest_neighbors = 0.19s
🔨 Preparing gallery (computing features)...


Computing features: 100%|██████████| 264/264 [00:25<00:00, 10.53it/s]


✅ Gallery prepared: 264 images
Computing queries features...


Computing features: 100%|██████████| 88/88 [00:08<00:00, 10.44it/s]


Elapsed time for evaluate = 88.11s
Elapsed time for inference = 0.08s
Elapsed time for find_nearest_neighbors = 0.69s
🔨 Preparing gallery (computing features)...


...

Elapsed time for evaluate = 322.72s
Elapsed time for inference = 0.61s
Elapsed time for find_nearest_neighbors = 1.24s
🔨 Preparing gallery (computing features)...


Computing features: 100%|██████████| 264/264 [08:23<00:00,  1.91s/it]


✅ Gallery prepared: 264 images
Computing queries features...


Computing features: 100%|██████████| 88/88 [02:48<00:00,  1.91s/it]


ZeroDivisionError: division by zero

In [3]:
df

,evaluation time,inference time,recall@1,recall@3,recall@10,store size (mb)
siamesedino,3.170333,0.190323,0.579545,0.727273,0.840909,6576
orb,88.769850,0.700401,0.829545,0.863636,0.886364,6576
0.5_orb+0.5_rapidocr,318.283803,1.562125,0.715909,0.761364,0.806818,6576


In [2]:
img1 = "/home/timothee/Documents/programmation/DIHT/data/original_data/002/IMG_20240220_175710405.jpg"
img2 = "/home/timothee/Documents/programmation/DIHT/data/original_data/002/IMG_20240220_225113575.jpg"
f1 = sift.get_features(img1)
f2 = sift.get_features(img2)

In [3]:
sift.compute_distance(f1, f2)

0.002506265664160401